# Suite ANY — Values of any schema

`Schemas.OfAny.Value` is the kind of a value of any schema: built with `schemas`, it is restricted to them, and with
none, it is unrestricted. A restricted value is wrapped by its kind and schema, `{"scalar": {"str": "x"}}`, or written
bare where exactly one of its schemas takes its plain form, and read back never guessing its schema; an
unrestricted one is tagged with its schema, a basic scalar's tag or a registered schema's name,
and is assigned as a value, built standalone through the store where it is not a scalar.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import document, raises, root_of, single


def install(txn):
    """Values of each kind, a value of any of three schemas and of any schema, and Card, holding them."""
    S = txn.Schemas
    return txn.uid("any.cards/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Phone": S.OfObject, "Code": S.OfUnion, "Stamp": S.OfIntersection, "Tags": S.OfContainer,
                  "Labels": S.OfContainer, "Cell": S.OfObject, "Free": S.OfAny, "Card": S.OfObject})
        .spec([
            lambda _: _.Phone.name("Phone").properties({"number": S.OfStr}),
            lambda _: _.Code.name("Code").branches([
                lambda t: t.name("n").schema([S.OfInt]),
                lambda t: t.name("s").schema([S.OfStr]),
            ]),
            lambda _: _.Stamp.name("Stamp").parts([
                lambda t: t.name("at").schema([S.OfInt]),
                lambda t: t.name("by").schema([S.OfStr]),
            ]),
            lambda _: _.Tags.name("Tags").index([S.OfInt]).element([S.OfStr]),
            lambda _: _.Labels.name("Labels").index([S.OfStr]).element([S.OfStr]),  # a dict by text: a mapping
            lambda _: _.Cell.name("Cell").properties({"r": S.OfInt, "c": S.OfInt}),
            lambda _: _.Free.name("Free"),
            lambda _: _.Card.name("Card").ref(True).properties({
                "one": S.OfAny(lambda a: a.schemas([S.OfInt, S.OfStr, _.Phone])),
                "free": _.Free,
                "many": S.OfContainer(lambda c: c.index([S.OfInt]).element([_.Free])),
                "tags": S.OfAny(lambda a: a.schemas([S.OfBool, _.Tags])),
            }),
        ])]))])


store = B.Store()
Int, Text, Float, Bytes, Bool = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema(), store.Schemas.OfFloat.as_schema(), store.Schemas.OfBytes.as_schema(), store.Schemas.OfBool.as_schema()  # the store's own
store.transaction(install)
Phone, Code, Stamp, Tags, Labels, Cell, Free, Card = (
    store.registered(name) for name in ("Phone", "Code", "Stamp", "Tags", "Labels", "Cell", "Free", "Card"))
One = Card.properties["one"].schema


def held(prop):
    """What a property holds, read through its visitor: a scalar, or a visitor's schema's name; None where absent."""
    found = []
    prop.as_value(lambda a: a.as_scalar(lambda s: found.extend([] if s.get() is None else [s.get()]) or s)
                  .as_object(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_union(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_intersection(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v)
                  .as_container(lambda v: found.extend([v.as_schema().name] if v.as_schema() else []) or v))
    return found[0] if found else None

## ANY-01 · An `OfAny` restricted to the schemas selected, or to none; schemas that share a plain form valid, told apart by the wrapper

In [ ]:
def pointing(txn):
    """Thing, a reference object, and Pointer, whose property holds an int or a Thing: a reference, which no property
    holds."""
    S = txn.Schemas
    return txn.uid("any.pointing/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Thing": S.OfObject, "Pointer": S.OfObject})
        .spec([
            lambda _: _.Thing.name("Thing").ref(True),
            lambda _: _.Pointer.name("Pointer").properties({"x": S.OfAny(lambda a: a.schemas([S.OfInt, _.Thing]))}),
        ])]))])


with store.transaction():  # the builders: values of any schema built alone, linked nowhere, validated as they are
    Grid = store.Schemas.OfContainer(lambda c: c.index([Cell]).element([Int]))  # a tensor: nested arrays, or flat with its shape
    assert type(One) is S.OfAny.Value and One.schemas == (Int, Text, Phone) and One.validate() == []
    assert Free.schemas == () and Free.validate() == [] and Card.validate() == []
    only = store.Schemas.OfAny(lambda a: a.schemas([Int]))  # one type is an OfAny still: where one type is meant, give it directly
    assert type(only) is S.OfAny.Value and only.schemas == (Int,) and store.Schemas.OfProperty(lambda p: p.schema([Int])).schema is Int
    assert (S.plain_forms(Float), S.plain_forms(Bytes), S.plain_forms(Labels), S.plain_forms(Tags), S.plain_forms(Grid)) == (
        {"float", "text"}, {"text"}, {"mapping"}, {"list"}, {"list", "mapping"})  # NaN is text, bytes base64, a tensor flat
    assert S.plain_forms(One) == {"int", "text", "mapping"} and S.plain_forms(Free) == {"mapping"}  # unrestricted: tagged
    assert S.plain_forms(S.OfScalar.Data(name="Word")) == {"bool", "int", "float", "text", "mapping", "list"}  # not a basic scalar
    assert S.OfAny.Value(schemas=(S.OfScalar.Data(),)).validate() == [
        "a schema: an unnamed scalar is not a basic scalar: a scalar is Schemas.OfBool, Schemas.OfInt, Schemas.OfFloat, Schemas.OfStr or Schemas.OfBytes"]
    # Schemas that share a plain form are valid: a value is wrapped by its kind and schema where it would be ambiguous.
    assert store.Schemas.OfAny(lambda a: a.schemas([Tags, Grid])).validate() == []
    assert store.Schemas.OfAny(lambda a: a.schemas([Cell, Labels])).validate() == []
    assert store.Schemas.OfAny(lambda a: a.schemas([store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("r").schema([Int])])),
                                                    Labels])).validate() == []
    assert One == One and One != Free  # schemas by identity
    Either = store.Schemas.OfUnion(lambda u: u.branches([
        lambda t: t.name("a").schema([Int]),
        lambda t: t.name("b").schema([Text]),
    ]))
    Both = store.Schemas.OfIntersection(lambda i: i.parts([
        lambda t: t.name("a").schema([Int]),
        lambda t: t.name("b").schema([Text]),
    ]))
    assert store.Schemas.OfAny(lambda a: a.schemas([Float, Text, Phone, Code, Labels, Either, Both, Tags])).validate() == []
    with raises(TypeError, match="link 'schema' cannot link 3"):  # the builders link schemas, nothing else
        store.Schemas.OfAny(lambda a: a.schemas([3]))
    assert S.OfAny.Value((3, One, S.OfScalar.Data(int, description=True), Grid)).validate() == [  # data they refuse to build
        "an OfAny holds schemas, got 3", "an OfAny holds schemas of other kinds, not an OfAny",
        "Schemas.OfInt: a description is text, got bool"]
    assert S.OfAny.Value((S.OfObject.Data(description=True), S.OfUnion.Data(description=True),  # unnamed: shown by their kind
                          S.OfIntersection.Data(description=True),
                          S.OfContainer.Data(index=Int, element=Int, description=True))).validate() == [
        "a value object: a description is text, got bool", "a union: a description is text, got bool",
        "a union: a union needs at least two branches", "an intersection: a description is text, got bool",
        "an intersection: an intersection needs at least two parts", "a container: a description is text, got bool"]
store.transaction(pointing)
assert store.registered("Pointer").validate() == [
    "property 'x': a reference object schema cannot be a property's schema"]  # a value only

## ANY-02 · Proxies: a restricted value written through the visitors as its schema is, inferred; an unrestricted one given as a value, built standalone where it is not a scalar

In [ ]:
with store.transaction():
    card = store.Card(lambda c: c.one(3).free("x"))
    assert (held(card.one), held(card.free)) == (3, "x") and card.one.get() == 3
    card = store.Card(lambda c: (
        c
        .one(lambda p: p.as_object(lambda o: o.number("1")))  # the one type of its kind
        .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a"))))
    ))
    assert (held(card.one), held(card.tags)) == ("Phone", "Tags") and card.one.get() is None
    with raises(TypeError, match="property 'one' does not hold a value of Tags"):
        store.Card(lambda c: c.one(store.Tags(lambda t: t)))
    with raises(TypeError, match="'Phone' is a value object schema; build a value through its owner, or standalone through "
                                "the store"):
        Proxies.OfObject.Builder(store, Phone, "Phone")
    with raises(ReadOnlyError, match="property 'one' does not hold a union value"):
        store.Card(lambda c: c.one(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.n(1)))))
    # Unrestricted: a scalar, or a value, standalone or another's; a Spec has no schema to build by.
    phone = store.Phone(lambda p: p.number("2"))
    tags = store.Tags(lambda t: t.at(0).as_scalar("t"))
    assert Proxies.owner_of(phone) is None and tags.as_schema() is Tags
    values = [phone, store.Code(lambda c: c.n(1)), store.Stamp(lambda s: s.at(1).by("me")), tags, card.one]
    assert [held(store.Card(lambda c: c.free(value)).free) for value in values] == ["Phone", "Code", "Stamp", "Tags", "Phone"]
    assert held(store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_object(card.one)))).free) == "Phone"
    fresh = store.Phone(lambda p: p.number("3"))
    first, second = store.Card(lambda c: c.free(fresh)), store.Card(lambda c: c.free(fresh))
    phones = []
    for each in (first, second):
        each.free.as_value(lambda a: a.as_object(lambda o: phones.append(o) or o))
    assert phones[0] is fresh and phones[1] is not fresh and Proxies.owner_of(fresh) is first  # adopted once, then copied
    unbuilt = "property 'free' holds a value of any schema: give a value, which carries its schema, not a Spec"
    with raises(ReadOnlyError, match=unbuilt):
        store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_object(lambda o: o.number("1")))))
    with raises(ReadOnlyError, match=unbuilt):
        store.Card(lambda c: c.free(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.remove(0)))))
    read = []  # a Spec of another kind than the value held reads it as empty, and writes nothing
    store.Card(lambda c: c.one(3).one(lambda p: p.as_value(lambda a: a.as_object(lambda o: read.append(o.as_schema()) or o))))
    assert read == [None]
    found = []
    card.free.as_value(lambda a: a.as_scalar(lambda s: found.append((s.as_schema(), s.get())) or s))
    assert found == [(Free, None)]  # nothing held: a scalar of any schema, absent
    listed = store.Card(lambda c: c.many(lambda p: p.as_value(lambda a: a.as_container(
        lambda c: c.at(0).as_scalar(1) and c.at(1).as_object(phone)))))
    elements = []
    listed.many.as_value(lambda a: a.as_container(lambda c: c.as_entries(elements.extend)))
    assert [index for index, _ in elements] == [0, 1]
    with raises(TypeError, match="property 'one' does not hold a value of a container"):
        store.Card(lambda c: c.one(listed.many))

## ANY-03 · Snapshots: a restricted value bare, an unrestricted one tagged, read back; malformed ones refused, with a path

In [ ]:
def spotted(txn):
    """Spot, a reference object, at a point, a value object of no name: not registered."""
    S = txn.Schemas
    return txn.uid("any.spotted/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Spot": S.OfObject})
        .spec([
            lambda _: _.Spot.name("Spot").ref(True).properties({"at": S.OfObject(lambda o: o.properties({"x": S.OfInt}))}),
        ])]))])


other = B.Store()  # another store of the same schemas
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own


def many(c):
    c.at(0).as_scalar(2.5)
    c.at(1).as_object(phone)
    c.at(3).as_container(tags)  # nothing at 2
    return c


with store.transaction():
    card = store.Card(lambda c: (
        c
        .one(lambda p: p.as_object(lambda o: o.number("1")))
        .free(store.Code(lambda x: x.s("c")))
        .many(lambda p: p.as_value(lambda a: a.as_container(many)))
        .tags(True)
    ))
written = root_of(Plain.ToPlain(store).OfObject(Card, card))
assert written == {"one": {"number": "1"}, "free": {"Code": {"s": "c"}}, "tags": True,
                   "many": [{"float": 2.5}, {"Phone": {"number": "2"}}, None, {"Tags": ["t"]}]}
for encode, decode in [(JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    text_ = encode(store).OfObject(Card, card)
    assert encode(other).OfObject(other.Card, decode(other).OfObject(other.Card, text_)) == text_
with store.transaction():
    assert root_of(Plain.ToPlain(store).OfObject(Card, store.Card(lambda c: c.one("s").free(b"\x01")))) == {
        "one": "s", "free": {"bytes": "AQ=="}}
assert Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) is not card  # counted: made anew
back = Plain.FromPlain(other).OfObject(other.Card, document(single("Card", {"one": 3, "tags": ["a"]})))
assert (held(back.one), held(back.tags)) == (3, "Tags")  # each read as the one type its plain form fits
assert held(Plain.FromPlain(other).OfObject(other.Card, document(single("Card", {"one": "s"}))).one) == "s"


def refused(values, message):
    at = '$.mutations[0].upsert.exists[0].spec[0]["0"][0].as_property.mutations[0].set.value'  # its one member's value
    with raises(DecodeError, match=at + message):
        Plain.FromPlain(other).OfObject(other.Card, document(single("Card", values)))


refused({"one": True}, ": a value of any schema takes the plain form of one of its schemas, got bool")
refused({"one": 2.5}, ": a value of any schema takes the plain form of one of its schemas, got float")
refused({"one": None}, ": a value of any schema takes the plain form of one of its schemas, got NoneType")
refused({"free": 3}, ": a value of any schema is {schema: value}, its schema a basic scalar's tag or a registered "
                     "schema's name")
refused({"free": {"Nope": 1}}, ".Nope: no schema registered as 'Nope'")
refused({"free": {"Card": {}}}, ".Card: 'Card' is not a value's schema")
refused({"free": {"int": "x"}}, ".int: expected int, got str")
with store.transaction():
    with raises(TypeError, match="a value of any schema holds one of its schemas; float is not one"):
        Plain.ToPlain(store).OfObject(Card, store.Card(lambda c: c.one(2.5)))  # builders do not check scalars' types
store.transaction(spotted)
with store.transaction():
    loose = store.Card(lambda c: c.free(store.Spot(lambda s: s.at(lambda p: p.as_object(lambda o: o.x(1)))).at))
    with raises(TypeError, match="a value of any schema is of a registered schema; None is not registered"):
        Plain.ToPlain(store).OfObject(Card, loose)

## ANY-04 · Validation and comparison under the schema a value holds; uniqueness by value

In [ ]:
def holding(txn):
    """Holds, whose entries hold a value of any schema (Free), and Holder."""
    S = txn.Schemas
    return txn.uid("any.holding/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Holder": S.OfObject})
        .spec([
            lambda _: _.Holds.name("Holds").links([lambda t: t.name("card"), lambda t: t.name("other")]).properties({
                "what": txn.Free, "note": S.OfStr}).uniques("note"),
            lambda _: _.Holder.name("Holder").ref(True).adjacencies({
                "holds": lambda a: a.relation([_.Holds]).me("card"),
                "held": lambda a: a.relation([_.Holds]).me("other"),
            }),
        ])]))])


store.transaction(holding)
other.transaction(holding)  # in other too, its own
Holder = store.registered("Holder")
with store.transaction():
    validate = Validators.Validate(store)
    assert validate(Card, card) == []
    assert validate(Card, store.Card(lambda c: c.one(2.5).many(lambda p: p.as_value(lambda a: a.as_container(
        lambda c: c.at(0).as_object(phone)))))) == ["Card#0.one: float is not one of its schemas"]
    assert validate(Card, loose) == ["Card#0.free: a value of any schema is of a registered schema; None is not registered"]
    assert validate(Card, store.Card(lambda c: c.free(store.Code(lambda x: x.n("x"))))) == ["Card#0.free.n: expected int, got str"]


    def compare(a, b):
        return C.OfObject(Card, a).compare(C.OfObject(Card, b))


    assert compare(store.Card(lambda c: c.one(3).free(phone)), store.Card(lambda c: c.one(3).free(store.Phone(lambda p: p.number("2"))))) == 0
    assert compare(store.Card(lambda c: c.one(3)), store.Card(lambda c: c.one("3"))) is None  # of different types: incomparable
    assert compare(store.Card(lambda c: c.free(phone)), store.Card(lambda c: c.free(store.Code(lambda x: x.s("2"))))) is None

    target = store.Holder(lambda h: h)
    holder = store.Holder(lambda h: h.holds(lambda a: (
        a
        .insert(lambda e: e.other(target).what(phone).note("a"))
        .insert(lambda e: e.other(target).what(store.Phone(lambda p: p.number("2"))).note("b"))
        .insert(lambda e: e.other(target).what(1).note("c"))
        .insert(lambda e: e.other(target).note("d"))
    )))
    assert validate(Holder, holder) == [
        "unique(note) violated: entries agreeing on ['card', 'other', 'what'] differ on ['note']"]  # equal values: one key
snapshot = Plain.ToPlain(store).Reachable(Holder, holder)
assert [e.get("what") for e in root_of(snapshot)["holds"]] == [{"Phone": {"number": "2"}}] * 2 + [{"int": 1}, None]
assert Plain.ToPlain(other).Reachable(other.Holder, Plain.FromPlain(other).Reachable(other.Holder, snapshot)) == snapshot
again = Plain.FromPlain(store).Reachable(Holder, snapshot)  # counted: made anew, with its entries
assert again is not holder and len(again.holds) == 4 and len(holder.holds) == 4
read = Plain.FromPlain(other).OfObject(other.Card, Plain.ToPlain(store).OfObject(Card, card))  # other's own Card
OtherCard = other.Card.as_schema()
with other.transaction():
    assert C.OfObject(OtherCard, read).compare(C.OfObject(OtherCard, other.Card(read))) == 0  # a container of values of any
    # type, element by element: an object and its clone

## ANY-05 · Snapshots write a value of any schema's schemas, linked, and read them back

In [ ]:
def typed_cards(txn):
    """Phone, and Card, holding an int, a text or a Phone, and a value of any schema."""
    S = txn.Schemas
    return txn.uid("any.typed/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Phone": S.OfObject, "Card": S.OfObject})
        .spec([
            lambda _: _.Phone.name("Phone").properties({"number": S.OfStr}),
            lambda _: _.Card.name("Card").ref(True).properties({
                "one": S.OfAny(lambda a: a.schemas([S.OfInt, S.OfStr, _.Phone])),
                "free": S.OfAny(lambda a: a),
            }),
        ])]))])


typed = B.Store()  # a snapshot holds every schema of its store: these, typed by the basic scalars
typed.transaction(typed_cards)
log = Plain.ToPlain(typed).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log  # read back, written again: the same
one, free = (back.registered("Card").properties[name].schema for name in ("one", "free"))
assert type(one) is S.OfAny.Value and one.schemas[2] is back.registered("Phone") and one.schemas[0] is back.Schemas.OfInt.as_schema()
assert free.schemas == ()

## ANY-06 · A restricted value never guessed: wrapped by its kind and schema, bare only where one of its schemas fits; the wrapper always read, malformed ones refused

In [ ]:
def mixing(txn):
    """Counts, a second list; Box, a value object whose one property is named as a kind is; and Mixed, holding values of
    `OfAny`s restricted to schemas that share plain forms."""
    S = txn.Schemas
    return txn.uid("any.mixing/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Counts": S.OfContainer, "Box": S.OfObject, "Mixed": S.OfObject})
        .spec([
            lambda _: _.Counts.name("Counts").index([S.OfInt]).element([S.OfInt]),
            lambda _: _.Box.name("Box").properties({"object": S.OfStr}),
            lambda _: _.Mixed.name("Mixed").ref(True).properties({
                "scalar": S.OfAny(lambda a: a.schemas([S.OfBool, S.OfInt, S.OfFloat, S.OfStr, S.OfBytes])),
                "value": S.OfAny(lambda a: a.schemas([S.OfInt, txn.Phone, txn.Code, txn.Stamp, txn.Labels, _.Box])),
                "lists": S.OfAny(lambda a: a.schemas([S.OfStr, txn.Tags, _.Counts])),
                "either": S.OfAny(lambda a: a.schemas([txn.Tags, txn.Labels])),  # two containers, of two plain forms
            }),
        ])]))])


store.transaction(mixing)
other.transaction(mixing)  # in other too, its own
Mixed = store.registered("Mixed")
Scalars, Values, Lists = (Mixed.properties[name].schema for name in ("scalar", "value", "lists"))
assert [t.validate() for t in (Scalars, Values, Lists)] == [[]] * 3  # schemas that share a plain form: told apart by the wrapper
to, back = Plain.ToPlain(store), Plain.FromPlain(store)
with store.transaction():
    # Bare where exactly one of its schemas takes the plain form written; else wrapped by its kind and schema.
    nan, inf = float("nan"), float("inf")
    for value, plain in [(True, True), (1, 1), (1.5, 1.5), ("x", {"scalar": {"str": "x"}}),
                         (b"\x01", {"scalar": {"bytes": "AQ=="}}), (nan, {"scalar": {"float": "NaN"}}),
                         (inf, {"scalar": {"float": "Infinity"}}), (-inf, {"scalar": {"float": "-Infinity"}})]:
        assert to(Scalars, value) == plain
        got = back(Scalars, plain)
        assert type(got) is type(value) and repr(got) == repr(value)
    phone_, code_, stamp_, labels_, box_ = (
        store.Phone(lambda p: p.number("1")), store.Code(lambda c: c.n(2)), store.Stamp(lambda s: s.at(3).by("me")),
        store.Labels(lambda l: l.at("k").as_scalar("v")), store.Box(lambda b: b.object("x")))
    tags_, counts = store.Tags(lambda t: t.at(0).as_scalar("t")), store.Counts(lambda c: c.at(0).as_scalar(5))
    for schema, value, plain in [
        (Values, 1, 1),
        (Values, phone_, {"object": {"Phone": {"number": "1"}}}),  # a value object, by its registered name
        (Values, code_, {"union": {"Code": {"n": 2}}}),
        (Values, stamp_, {"intersection": {"Stamp": {"at": 3, "by": "me"}}}),
        (Values, labels_, {"container": {"k": "v"}}),  # the one container schema: its kind alone
        (Values, box_, {"object": {"Box": {"object": "x"}}}),
        (Lists, "x", "x"),
        (Lists, tags_, {"container": {"Tags": ["t"]}}),  # one of two containers: its kind and its schema
        (Lists, counts, {"container": {"Counts": [5]}}),
    ]:
        assert to(schema, value) == plain
        assert to(schema, back(schema, plain)) == plain
    assert back(Values, {"object": {"Phone": {"number": "1"}}}).as_schema() is Phone
    assert back(Lists, {"container": {"Counts": [5]}}).as_schema().name == "Counts"
    # A mapping that would read as a wrapper is wrapped, though it is the one mapping; another is bare.
    assert to(store.Schemas.OfAny(lambda a: a.schemas([Int, store.registered("Box")])), box_) == {
        "object": {"Box": {"object": "x"}}}
    assert to(store.Schemas.OfAny(lambda a: a.schemas([Int, Phone])), phone_) == {"number": "1"}
    # The reader always takes the wrapper, and the bare shorthand only where exactly one of its schemas fits it.
    assert back(Scalars, {"scalar": {"int": 7}}) == 7 and back(Scalars, {"scalar": {"bool": False}}) is False
    assert back(Scalars, {"scalar": {"float": 1.5}}) == 1.5 and back(Values, 7) == 7
    assert back(Values, {"scalar": {"int": 7}}) == 7 and back(Lists, {"scalar": {"str": "y"}}) == "y"
    anonymous = store.Schemas.OfAny(lambda a: a.schemas([store.Schemas.OfObject(lambda o: o.properties({"x": Int})), Phone]))
    assert back(anonymous, {"object": {"Phone": {"number": "4"}}}).as_schema() is Phone  # a schema of no name: none names it
    unnamed = store.Schemas.OfAny(lambda a: a.schemas([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Text])), Labels]))
    with raises(TypeError, match="a standalone value is built through the store's builder of its schema"):
        back(unnamed, ["t"])  # bare, the one list; one of two containers, of no name: not built standalone
for plain, message in [
    ((Scalars, "x"), "$: several of this value's schemas take a bare str: it is wrapped by its kind and schema, never guessed"),
    ((Values, {"number": "1"}), "$: several of this value's schemas take a bare dict"),
    ((Lists, ["t"]), "$: several of this value's schemas take a bare list"),
    ((Scalars, [1]), "$: a value of any schema takes the plain form of one of its schemas, got list"),
    ((Scalars, {"tensor": [1]}), "$: a value of any schema is wrapped by its kind, one of ['container', 'intersection', "
                                 "'object', 'scalar', 'union'], got 'tensor'"),
    ((Lists, {"object": {"Phone": {}}}), "$.object: no object schema is one of this value's schemas"),
    ((Lists, {"scalar": {"int": 1}}), "$.scalar.int: 'int' is not one of this value's scalar schemas"),
    ((Values, {"object": {"Cell": {"r": 1}}}), "$.object.Cell: 'Cell' is not one of this value's object schemas"),
    ((Values, {"union": "x"}), "$.union: a value wrapped as 'union' is {schema: value}"),
    ((Values, {"union": {"Code": {}, "Stamp": {}}}), "$.union: a value wrapped as 'union' is {schema: value}"),
    ((Scalars, {"scalar": {"str": 1}}), "$.scalar.str: expected str, got int"),
]:
    with raises(DecodeError, match=message):
        with store.transaction():
            back(*plain)
with store.transaction():  # in a document, through JSON and YAML: wrapped where its schemas share a plain form
    mixed = store.Mixed(lambda m: m.scalar("x").value(store.Code(lambda c: c.s("y")))
                        .lists(store.Tags(lambda t: t.at(0).as_scalar("u"))).either(store.Tags(lambda t: t.at(0).as_scalar("w"))))
assert root_of(Plain.ToPlain(store).OfObject(Mixed, mixed)) == {
    "scalar": {"scalar": {"str": "x"}}, "value": {"union": {"Code": {"s": "y"}}}, "lists": {"container": {"Tags": ["u"]}},
    "either": ["w"]}  # bare: of two containers, the one list
for encode, decode in [(JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    text_ = encode(store).OfObject(Mixed, mixed)
    assert encode(other).OfObject(other.Mixed, decode(other).OfObject(other.Mixed, text_)) == text_